# 21 CATBOOST: CatBoost with early stopping on the evaluation measure

With CatBoost's built-in `PRAUC` as the early-stopping metric, most resampled CatBoost fits at D5 and W5 stop within five iterations (`08_SEEDS`). The validation block is not short of positives (60 positive runs at D5 and W5, 56 at W60), and XGBoost under the same resampling runs for many more iterations. The cause is the metric: after one or two trees the model gives only a few dozen distinct scores over the validation block, and on such heavily tied scores the built-in `PRAUC` is unreliable, so early stopping locks onto a false peak. Under default class weighting the built-in metric is also weighted by class, unlike the PR-AUC reported in the paper. Section 2 reproduces the diagnosis.

Here early stopping uses the measure reported in the paper: the average precision of scikit-learn on the unweighted validation block, passed to CatBoost as a custom metric. Everything else is as in `08_SEEDS`: data, partitions, patience (100), maximum iterations (800), the two-stage fit, sampling strategies and seeds (42, 1, 2, 3, 4).

* **What runs.** All 120 CatBoost fits: the pooled target under the five strategies at the four windows, and the other messages under default weighting at the four windows, five seeds each. XGBoost and Logistic Regression are read from the results of `08_SEEDS`.
* **Environment.** Python 3.14.3 (the labelled telemetry needs pandas 3). Each fit is appended to the results file as it finishes; a rerun skips completed fits and repeats failed ones. The full run takes a few hours.

Outputs (`OUTPUT/v4/`): `rev_catboost_results.csv` (one row per fit), `rev_catboost_diagnosis.csv` (Section 2), `rev_catboost_summary.csv` (this notebook against the built-in stopping metric of `08_SEEDS`, with XGBoost), `rev_catboost_instability.csv`, `rev_catboost_best.csv`.

Paper: Sections 2.7.1 and 3.2; Tables 2 and 6 to 8.

In [ ]:
import os, sys, time, warnings
import numpy as np, pandas as pd
import sklearn, xgboost, catboost, imblearn
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_recall_curve
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
warnings.filterwarnings("ignore")

A2 = "../ANALYSIS2"
env = os.environ.get
CFG = dict(
    labelled   = os.environ.get("REV_BUILDDIR", "build") + "/telemetry_labelled_v4.pkl",
    episodes   = os.environ.get("REV_BUILDDIR", "build") + "/fault_episodes_v4.csv",
    ref_pooled = f"{A2}/OUTPUT/classification_results_v3.csv",
    ref_groups = f"{A2}/OUTPUT/blower_performance_check.csv",
    outdir     = env("REV_OUTDIR", "OUTPUT") + "/v4",
    blower     = "AUX CAP MACH RM BLOW1 NOT OFF",
    horizons   = env("REV_HORIZONS", "W60,W300,W5,D5").split(","),   # run order
    seeds      = [int(s) for s in env("REV_SEED_LIST", "42,1,2,3,4").split(",")],
    samplers   = env("REV_SAMPLERS", "Default (class weight),Undersampling,Random Oversampling,SMOTE,ADASYN").split(","),
    windows    = {"D5": (-5, 5), "W5": (-5, -1), "W60": (-60, -1), "W300": (-300, -1)},
    patience   = 100, test_frac = 0.20, val_frac = 0.20,
    recall_points = [0.25, 0.50],
)
CFG["results"] = f"{CFG['outdir']}/rev_catboost_results.csv"
CFG["old_results"] = f"{CFG['outdir']}/rev_seeds_results.csv"
NUMERIC = ["KMH", "DIST_M", "KV", "AMP", "KWH", "BV", "PRESSURE", "TRACTION_PCT", "BG1_KN", "BG2_KN", "BG1_DCLV", "BG2_DCLV"]
WHEEL_SLIP = ["Wheel_Slip_TM2", "Wheel_Slip_TM3", "Wheel_Slip_TM4", "Wheel_Slip_TM5", "Wheel_Slip_TM6"]
PRIMARY = NUMERIC + WHEEL_SLIP
os.makedirs(CFG["outdir"], exist_ok=True)
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| sklearn", sklearn.__version__,
      "| xgboost", xgboost.__version__, "| catboost", catboost.__version__, "| imblearn", imblearn.__version__)
print("Horizons", CFG["horizons"], "| seeds", CFG["seeds"])

## 1. Functions (as in `08_SEEDS`, with the CatBoost stopping metric replaced)

In [ ]:
class SklearnAP:
    """CatBoost custom metric: scikit-learn average precision on the unweighted
    validation block, the PR-AUC reported in the paper. Raw scores are used; average
    precision depends only on their order, so the logistic transform is not needed."""
    def is_max_optimal(self):
        return True
    def get_final_error(self, error, weight):
        return error
    def evaluate(self, approxes, target, weight):
        return float(average_precision_score(np.asarray(target), np.asarray(approxes[0]))), 1.0

def chrono_cut(y, frac):
    n = len(y)
    i = int(n * (1 - frac))
    while i < n - 1 and y[i] == 1 and y[i - 1] == 1:
        i += 1
    return i

def three_way_split(y, cfg):
    n = len(y)
    t = int(n * (1 - cfg["test_frac"]))
    while t < n - 1 and y[t] == 1:
        t += 1
    v = chrono_cut(y[:t], cfg["val_frac"])
    return np.arange(v), np.arange(v, t), np.arange(t), np.arange(t, n)

SAMPLERS = {
    "Default (class weight)": None,
    "Undersampling":          lambda s: RandomUnderSampler(random_state=s),
    "Random Oversampling":    lambda s: RandomOverSampler(random_state=s),
    "SMOTE":                  lambda s: SMOTE(random_state=s),
    "ADASYN":                 lambda s: ADASYN(random_state=s),
}

def make_model(name, seed, balanced, y, n_iter=None):
    if name == "Logistic Regression":
        return LogisticRegression(max_iter=2000, random_state=seed, class_weight="balanced" if balanced else None)
    if name == "XGBoost":
        spw = float((y == 0).sum() / max((y == 1).sum(), 1)) if balanced else 1.0
        kw = dict(n_estimators=600, early_stopping_rounds=CFG["patience"]) if n_iter is None \
             else dict(n_estimators=n_iter)
        return XGBClassifier(learning_rate=0.05, max_depth=6, subsample=0.8,
                             colsample_bytree=0.8, eval_metric="aucpr",
                             tree_method="hist", scale_pos_weight=spw,
                             random_state=seed, n_jobs=-1, **kw)
    if name == "CatBoost":
        kw = dict(iterations=800, early_stopping_rounds=CFG["patience"]) if n_iter is None \
             else dict(iterations=n_iter)
        if n_iter is None:
            kw["eval_metric"] = SklearnAP()          # stopping metric = the paper's PR-AUC
        return CatBoostClassifier(learning_rate=0.05, depth=6, auto_class_weights="Balanced" if balanced else None, random_seed=seed, verbose=0, **kw)

def resample(sampler, X, y, seed):
    return (X, y) if sampler is None else SAMPLERS[sampler](seed).fit_resample(X, y)

def fit_two_stage(name, sampler, Xa, ya, Xv, yv, Xtr, ytr, seed):
    balanced = sampler is None
    if name == "Logistic Regression":
        sc = StandardScaler().fit(Xtr)
        Xs, ys = resample(sampler, sc.transform(Xtr), ytr, seed)
        m = make_model(name, seed, balanced, ys).fit(Xs, ys)
        return m, sc, None
    Xs, ys = resample(sampler, Xa, ya, seed)
    m1 = make_model(name, seed, balanced, ys)
    if name == "XGBoost":
        m1.fit(Xs, ys, eval_set=[(Xv, yv)], verbose=False)
        n_iter = int(m1.best_iteration) + 1
    else:
        m1.fit(Xs, ys, eval_set=(Xv, yv), use_best_model=True)
        n_iter = int(m1.get_best_iteration()) + 1
    Xs, ys = resample(sampler, Xtr, ytr, seed)
    m2 = make_model(name, seed, balanced, ys, n_iter=n_iter).fit(Xs, ys)
    return m2, None, n_iter

def score(m, sc, X):
    return m.predict_proba(sc.transform(X) if sc is not None else X)[:, 1]

def evaluate(y, s, recall_points):
    p, r, _ = precision_recall_curve(y, s)
    out = dict(pr_auc=average_precision_score(y, s))
    for rp in recall_points:
        out[f"precision_at_recall_{rp:.2f}"] = float(p[r >= rp].max())
    return out

def coverage(ts, onsets, lo, hi):
    d = np.zeros(len(ts) + 1, dtype=np.int32)
    o = pd.DatetimeIndex(onsets).values.astype("datetime64[s]")
    a = np.searchsorted(ts, o + np.timedelta64(lo, "s"), side="left")
    b = np.searchsorted(ts, o + np.timedelta64(hi, "s"), side="right")
    np.add.at(d, a, 1); np.add.at(d, b, -1)
    return np.cumsum(d)[:-1]

In [ ]:
tel = pd.read_pickle(CFG["labelled"]).sort_values("DATETIME").reset_index(drop=True)
if env("REV_SAMPLE"):                     # trial runs only
    tel = tel.iloc[:int(len(tel) * float(env("REV_SAMPLE")))].copy()
    print(f"TRIAL RUN on {len(tel):,} rows")
ts = tel["DATETIME"].values.astype("datetime64[s]")
ep = pd.read_csv(CFG["episodes"], parse_dates=["onset", "recovery"])
ep = ep[(ep.onset >= tel.DATETIME.min()) & (ep.onset <= tel.DATETIME.max())]
is_b = ep.fault.eq(CFG["blower"]).values
X_all = tel[PRIMARY].astype("float32").fillna(0).values

def target(h, kind):
    y0 = tel[h].astype(int).values
    if kind == "pooled":
        return y0
    lo, hi = CFG["windows"][h]
    cov_b = coverage(ts, ep.onset[is_b], lo, hi)
    cov_o = coverage(ts, ep.onset[~is_b], lo, hi)
    return np.where((y0 == 1) & (cov_b > 0) & (cov_o == 0), -1, y0)   # other labels

print(f"{len(tel):,} rows; {len(ep)} onsets ({is_b.sum()} blower, {(~is_b).sum()} other)")

## 2. Diagnosis, reproduced (seed 42, W5, undersampling)

One inner-block fit of 200 trees without early stopping, scored on the validation block after each of the first trees by CatBoost's built-in `PRAUC` and by scikit-learn's average precision. About a minute.

In [ ]:
y = tel["W5"].astype(int).values
keep = y != -1
Xd, yd = X_all[keep], y[keep]
ia, iv, itr, ite = three_way_split(yd, CFG)
Xs, ys = RandomUnderSampler(random_state=42).fit_resample(Xd[ia], yd[ia])
md_ = CatBoostClassifier(learning_rate=0.05, depth=6, iterations=200, eval_metric="PRAUC",
                         random_seed=42, verbose=0).fit(Xs, ys, eval_set=(Xd[iv], yd[iv]), use_best_model=False)
builtin = md_.get_evals_result()["validation"]["PRAUC"]
rows = []
for n in [1, 2, 3, 5, 10, 25, 50, 100, 200]:
    s_ = md_.predict_proba(Xd[iv], ntree_end=n)[:, 1]
    rows.append(dict(trees=n, distinct_scores=len(np.unique(s_)), catboost_PRAUC=round(builtin[n - 1], 4),
                     sklearn_AP=round(average_precision_score(yd[iv], s_), 4),
                     test_AP=round(average_precision_score(yd[ite], md_.predict_proba(Xd[ite], ntree_end=n)[:, 1]), 4)))
diag = pd.DataFrame(rows)
print(diag.to_string(index=False))
diag.to_csv(f"{CFG['outdir']}/rev_catboost_diagnosis.csv", index=False)
print(f"\nBuilt-in PRAUC peaks after {int(np.argmax(builtin)) + 1} trees; "
      f"validation AP (scikit-learn) after the same number: "
      f"{average_precision_score(yd[iv], md_.predict_proba(Xd[iv], ntree_end=int(np.argmax(builtin)) + 1)[:, 1]):.4f}")
del Xd, yd, Xs, ys, md_

## 3. Run the 120 CatBoost fits (resumable)

In [ ]:
jobs = []
for h in CFG["horizons"]:
    for seed in CFG["seeds"]:
        for s in CFG["samplers"]:
            jobs.append(("pooled", h, s, "CatBoost", seed))
        jobs.append(("other labels", h, "Default (class weight)", "CatBoost", seed))

KEY = ["target", "horizon", "sampling", "model", "seed"]
done = set()
if os.path.exists(CFG["results"]):
    prev = pd.read_csv(CFG["results"])
    prev = prev[prev.status == "ok"].copy()          # failed fits are run again
    prev["seed"] = prev["seed"].astype(int)          # "42", not "42.0"
    done = set(map(tuple, prev[KEY].astype(str).values))
todo = [j for j in jobs if tuple(map(str, j)) not in done]
print(f"{len(jobs)} fits in total, {len(jobs) - len(todo)} already done, {len(todo)} to run")

In [ ]:
cache = {}
t_all = time.time()
for k, (kind, h, sname, mname, seed) in enumerate(todo, 1):
    if (kind, h) not in cache:
        cache.clear()
        y = target(h, kind)
        keep = y != -1
        X, y = X_all[keep], y[keep]
        cache[(kind, h)] = (X, y, three_way_split(y, CFG))
    X, y, (ia, iv, itr, ite) = cache[(kind, h)]
    t0 = time.time()
    try:
        m, sc, n_iter = fit_two_stage(mname, None if sname.startswith("Default") else sname, X[ia], y[ia], X[iv], y[iv], X[itr], y[itr], seed)
        r = evaluate(y[ite], score(m, sc, X[ite]), CFG["recall_points"])
        status = "ok"
    except Exception as e:
        r = dict(pr_auc=np.nan, **{f"precision_at_recall_{rp:.2f}": np.nan for rp in CFG["recall_points"]})
        n_iter, status = None, f"FAILED: {e}".replace("\n", " ")[:300]   # same columns as a finished fit
    base = y[ite].mean()
    r.update(target=kind, horizon=h, sampling=sname, model=mname, seed=seed, n_iter=n_iter, baseline=base, lift=r["pr_auc"] / base, seconds=round(time.time() - t0, 1), status=status)
    pd.DataFrame([r]).to_csv(CFG["results"], mode="a", index=False, header=not os.path.exists(CFG["results"]))
    print(f"[{k}/{len(todo)}] {kind:12s} {h:5s} {sname:22s} {mname:19s} seed {seed:2d}  "
          f"PR-AUC {r['pr_auc']:.4f}  iters {n_iter}  {r['seconds']:.0f} s", flush=True)
print(f"Finished in {(time.time() - t_all) / 60:.1f} min")

## 4. Built-in against evaluation stopping metric

Mean ± SD over the five seeds, iterations selected and the share of fits stopped within five iterations, for each target, window and strategy: "old" is CatBoost with the built-in `PRAUC` (`08_SEEDS`), "new" this notebook. XGBoost (`08_SEEDS`) is shown for reference.

In [ ]:
new = pd.read_csv(CFG["results"]); new = new[new.status == "ok"]
old = pd.read_csv(CFG["old_results"]); old = old[old.status == "ok"]
hz = [h for h in ["D5", "W5", "W60", "W300"] if h in new.horizon.unique()]

def agg(d):
    return d.groupby(["target", "horizon", "sampling"]).agg(
        mean=("pr_auc", "mean"), sd=("pr_auc", "std"), iters_min=("n_iter", "min"),
        iters_max=("n_iter", "max"), early_pct=("n_iter", lambda s: 100 * (s <= 5).mean()))
a_old = agg(old[old.model == "CatBoost"]).add_prefix("old_")
a_new = agg(new).add_prefix("new_")
a_xgb = agg(old[old.model == "XGBoost"])[["mean", "sd"]].add_prefix("xgb_")
summ = a_new.join(a_old).join(a_xgb)
summ.round(4).to_csv(f"{CFG['outdir']}/rev_catboost_summary.csv")

fmt = lambda m, s: f"{m:.3f} ± {0 if pd.isna(s) else s:.3f}"
show = pd.DataFrame({
    "CatBoost old": [fmt(r.old_mean, r.old_sd) for r in summ.itertuples()],
    "CatBoost new": [fmt(r.new_mean, r.new_sd) for r in summ.itertuples()],
    "XGBoost": [fmt(r.xgb_mean, r.xgb_sd) for r in summ.itertuples()],
    "iters old": [f"{int(r.old_iters_min)}–{int(r.old_iters_max)}" for r in summ.itertuples()],
    "iters new": [f"{int(r.new_iters_min)}–{int(r.new_iters_max)}" for r in summ.itertuples()],
    "≤5 old %": summ.old_early_pct.round(0).values, "≤5 new %": summ.new_early_pct.round(0).values,
}, index=summ.index)
pd.set_option("display.width", 250)
print(show.to_string())

## 5. Instability across strategies and seeds, and the best model per window

In [ ]:
pb = pd.concat([old[(old.target == "pooled") & (old.model == "XGBoost")], new[new.target == "pooled"]])
across_strat = (pb.groupby(["model", "horizon", "seed"])["pr_auc"].agg(lambda s: s.max() - s.min())
                  .groupby(["model", "horizon"]).mean().rename("range across strategies (mean over seeds)"))
across_seed = (pb.groupby(["model", "horizon", "sampling"])["pr_auc"].agg(lambda s: s.max() - s.min())
                 .groupby(["model", "horizon"]).mean().rename("range across seeds (mean over strategies)"))
iters = (pb.groupby(["model", "horizon"])["n_iter"].agg(lambda s: f"{int(s.min())}–{int(s.max())}").rename("iterations selected"))
early = (pb.assign(e=pb.n_iter <= 5).groupby(["model", "horizon"])["e"].mean().mul(100).round(1)
           .rename("fits stopped within 5 iterations (%)"))
inst = pd.concat([across_strat, across_seed, iters, early], axis=1).reindex(hz, level=1)
print(inst.round(3).to_string())
inst.to_csv(f"{CFG['outdir']}/rev_catboost_instability.csv")

allp = pd.concat([old[(old.target == "pooled") & (old.model != "CatBoost")], new[new.target == "pooled"]])
best = (allp.sort_values("pr_auc", ascending=False).groupby(["horizon", "seed"]).first()
            .reset_index()[["horizon", "seed", "model", "sampling", "pr_auc"]])
best["pr_auc"] = best.pr_auc.round(3)
print()
print(best.pivot(index="seed", columns="horizon", values="model").reindex(columns=hz).to_string(), "\n")
print(best.pivot(index="seed", columns="horizon", values="pr_auc").reindex(columns=hz).to_string())
best.to_csv(f"{CFG['outdir']}/rev_catboost_best.csv", index=False)

means = allp.groupby(["horizon", "model", "sampling"]).pr_auc.mean()
print("\nBest five-seed mean per window:")
for h in hz:
    k = means.loc[h].idxmax()
    print(f"  {h:5s} {k[0]:19s} {k[1]:22s} {means.loc[h].max():.3f}")

___
**END**